In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Pasul 1: Definirea modelelor Pydantic pentru output-uri structurate

Aceste modele definesc **schema** pe care agenții o vor returna. Utilizarea `response_format` cu Pydantic asigură:
- ✅ Extracție de date sigură din punct de vedere al tipului
- ✅ Validare automată
- ✅ Fără erori de parsare din răspunsuri text liber
- ✅ Rutare condiționată ușoară bazată pe câmpuri


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Pasul 2: Creează Instrumentul de Rezervare a Hotelurilor

Acest instrument este ceea ce va apela **availability_agent** pentru a verifica dacă camerele sunt disponibile. Folosim decoratorul `@ai_function` pentru:
- A converti o funcție Python într-un instrument apelabil de AI
- A genera automat schema JSON pentru LLM
- A gestiona validarea parametrilor
- A permite apelarea automată de către agenți

Pentru acest demo:
- **Stockholm, Seattle, Tokyo, Londra, Amsterdam** → Au camere ✅
- **Toate celelalte orașe** → Nu au camere ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Pasul 3: Definește Funcțiile de Condiție pentru Rutare

Aceste funcții inspectează răspunsul agentului și decid ce traseu să urmeze în fluxul de lucru.

**Model Cheie:**
1. Verifică dacă mesajul este `AgentExecutorResponse`
2. Analizează rezultatul structurat (model Pydantic)
3. Returnează `True` sau `False` pentru a controla rutarea

Fluxul de lucru va evalua aceste condiții pe **muchiile** grafului pentru a decide ce executor să invoce în continuare.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## Pasul 4: Creează un executor personalizat pentru afișare

Executorii sunt componente ale workflow-ului care realizează transformări sau efecte secundare. Folosim decoratoriul `@executor` pentru a crea un executor personalizat care afișează rezultatul final.

**Concepte cheie:**
- `@executor(id="...")` - Înregistrează o funcție ca executor în workflow
- `WorkflowContext[Never, str]` - Sugestii de tip pentru intrare/ieșire
- `ctx.yield_output(...)` - Returnează rezultatul final al workflow-ului


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## Pasul 5: Încarcă variabilele de mediu

Configurează clientul LLM. Acest exemplu funcționează cu:
- **Modelele GitHub** (nivel gratuit cu token GitHub)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Pasul 6: Creează agenți AI cu ieșiri structurate

Creăm **trei agenți specializați**, fiecare încapsulat într-un `AgentExecutor`:

1. **availability_agent** - Verifică disponibilitatea hotelului folosind unealta
2. **alternative_agent** - Sugerează orașe alternative (când nu sunt camere)
3. **booking_agent** - Încurajează rezervarea (când sunt camere disponibile)

**Caracteristici cheie:**
- `tools=[hotel_booking]` - Furnizează unealta agentului
- `response_format=PydanticModel` - Impune o ieșire JSON structurată
- `AgentExecutor(..., id="...")` - Învelește agentul pentru utilizarea în fluxul de lucru


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Pasul 7: Construiește fluxul de lucru cu muchii condiționale

Acum folosim `WorkflowBuilder` pentru a construi graful cu rutare condițională:

**Structura fluxului de lucru:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**Metode cheie:**
- `.set_start_executor(...)` - Setează punctul de intrare
- `.add_edge(from, to, condition=...)` - Adaugă muchie condițională
- `.build()` - Finalizează fluxul de lucru


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## Pasul 8: Rulați Cazul de Test 1 - Oraș FĂRĂ Disponibilitate (Paris)

Să testăm calea **fără disponibilitate** solicitând hoteluri în Paris (care nu are camere în simularea noastră).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Pasul 9: Rulează Cazul de Test 2 - Oraș CU Disponibilitate (Stockholm)

Acum să testăm calea **disponibilitate** cerând hoteluri în Stockholm (care are camere în simularea noastră).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## Aspecte esențiale și pași următori

### ✅ Ce ați învățat:

1. **Pattern-ul WorkflowBuilder**
   - Folosiți `.set_start_executor()` pentru a defini punctul de intrare
   - Folosiți `.add_edge(from, to, condition=...)` pentru rutare condiționată
   - Apelați `.build()` pentru a finaliza workflow-ul

2. **Rutare condiționată**
   - Funcțiile condiționale inspectează `AgentExecutorResponse`
   - Parcurgeți output-uri structurate pentru a lua decizii de rutare
   - Returnați `True` pentru a activa o muchie, `False` pentru a o sări

3. **Integrarea uneltelor**
   - Folosiți `@ai_function` pentru a converti funcții Python în unelte AI
   - Agenții apelează automat uneltele când este necesar
   - Uneltele returnează JSON pe care agenții îl pot parsa

4. **Output-uri structurate**
   - Folosiți modele Pydantic pentru extragerea de date sigură din punct de vedere al tipurilor
   - Stabiliți `response_format=MyModel` când creați agenții
   - Parsarea răspunsurilor cu `Model.model_validate_json()`

5. **Executori personalizați**
   - Folosiți `@executor(id="...")` pentru a crea componente workflow
   - Executorii pot transforma date sau efectua efecte secundare
   - Folosiți `ctx.yield_output()` pentru a produce rezultate ale workflow-ului

### 🚀 Aplicații reale:

- **Rezervare călătorii**: Verificați disponibilitatea, sugerați alternative, comparați opțiuni
- **Serviciu clienți**: Rutează în funcție de tipul problemei, sentiment, prioritate
- **Comerț electronic**: Verificați inventarul, sugerați alternative, procesați comenzi
- **Moderare conținut**: Rutează în funcție de scorurile de toxicitate, reclamanți
- **Fluxuri de aprobare**: Rutează în funcție de sumă, rolul utilizatorului, nivel de risc
- **Procesare în mai multe etape**: Rutează în funcție de calitatea, completitudinea datelor

### 📚 Pași următori:

- Adăugați condiții mai complexe (criterii multiple)
- Implementați bucle cu managementul stării workflow-ului
- Adăugați sub-workflow-uri pentru componente reutilizabile
- Integrați cu API-uri reale (rezervări hotel, sisteme de inventar)
- Adăugați tratarea erorilor și căi alternative
- Vizualizați workflow-urile cu uneltele de vizualizare integrate


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
